# Importing Stuff

In [16]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re

# Opening the dataframe

In [2]:
# Modify directory if needed
train = pd.read_csv("train.tsv", delimiter = '\t')
valid = pd.read_csv("valid.tsv", delimiter = '\t')
test = pd.read_csv("test.tsv", delimiter = '\t')

In [3]:
# adding column names
column_names = [
    'id', 'label', 'statement', 'subject', 'speaker', 'speaker_job_title', 'state_info',
    'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts',
    'mostly_true_counts', 'pants_on_fire_counts', 'context'
]

train.columns = column_names
valid.columns = column_names
test.columns = column_names

In [4]:
# combining all data frames
df = pd.concat([train, valid, test], axis=0).reset_index(drop=True)

df.head()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context
0,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
1,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
2,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
3,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN
4,12465.json,true,The Chicago Bears have had more starting quart...,education,robin-vos,Wisconsin Assembly speaker,Wisconsin,republican,0.0,3.0,2.0,5.0,1.0,a an online opinion-piece


In [18]:
df.tail()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,date
12783,7334.json,half-true,Says his budget provides the highest state fun...,education,rick-scott,Governor,Florida,republican,28.0,23.0,38.0,34.0,7.0,a news conference,NaT
12784,9788.json,barely-true,Ive been here almost every day.,"civil-rights,crime,criminal-justice",jay-nixon,Governor,Missouri,democrat,2.0,0.0,0.0,1.0,0.0,"on ABC's ""This Week""",NaT
12785,10710.json,barely-true,"In the early 1980s, Sen. Edward Kennedy secret...","bipartisanship,congress,foreign-policy,history",mackubin-thomas-owens,"senior fellow, Foreign Policy Research Institute",Rhode Island,columnist,1.0,0.0,0.0,0.0,0.0,a commentary in The Providence Journal,NaT
12786,3186.json,barely-true,Says an EPA permit languished under Strickland...,"environment,government-efficiency",john-kasich,"Governor of Ohio as of Jan. 10, 2011",Ohio,republican,9.0,8.0,10.0,18.0,3.0,a news conference,NaT
12787,6743.json,false,Says the governor is going around the state ta...,"state-budget,state-finances,taxes",john-burzichelli,NaN,NaN,democrat,1.0,1.0,0.0,0.0,0.0,an interview with NJToday,NaT


# Webscraping


## Add a date column

In [25]:
df['date'] = pd.NaT  # Initialize 'date' column with NaT (Not a Time)
df['date'] = pd.to_datetime(df['date'])  # Convert 'date' column to datetime type

In [26]:
df.head()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,date
0,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,NaT
1,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,NaT
2,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release,NaT
3,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN,NaT
4,12465.json,true,The Chicago Bears have had more starting quart...,education,robin-vos,Wisconsin Assembly speaker,Wisconsin,republican,0.0,3.0,2.0,5.0,1.0,a an online opinion-piece,NaT


## Webscraping - in batches

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}
dates = []

def process_batch(batch):
    batch_dates = []
    for _, row in batch.iterrows():
        statement = row["statement"]
        speaker = row["speaker"]

        # construct search URL
        search_url = f"https://www.politifact.com/search/?q={statement}+{speaker}"
        response = requests.get(search_url, headers=headers)

        soup = BeautifulSoup(response.text, "html.parser")

        # find the first search result container
        first_result = soup.find("div", class_="m-result__content")

        if first_result:
            date_element = first_result.find("div", class_="c-textgroup__author") 
            raw_date_text = date_element.text.strip().split("stated on")[-1].strip() if date_element else "Not found"

            # extract only the date using regex
            match = re.search(r"([A-Za-z]+ \d{1,2}, \d{4})", raw_date_text)
            clean_date = match.group(0) if match else "Not found"
        else:
            clean_date = "Not found"

        batch_dates.append(clean_date)
        print(f"Date found: {clean_date}")
    
    return batch_dates

# Number of rows to process per batch
batch_size = 100

# Find rows where 'date' is missing or "Not found"
missing_date_mask = df["date"].isna() | (df["date"] == "Not found")
rows_to_process = df[missing_date_mask]

# Loop through the dataframe in batches of 100
for start in range(0, len(rows_to_process), batch_size):
    end = start + batch_size
    batch = df.iloc[start:end]
    
    # Process the batch
    batch_dates = process_batch(batch)
    
    # Add the dates directly into the DataFrame immediately after processing the batch
    df.iloc[start:end, df.columns.get_loc('date')] = batch_dates

    # Introduce a delay to avoid getting blocked
    print(f"Processed rows {start + 1} to {end}")
    time.sleep(5)  # Delay between batches



Date found: February 04, 2015
Date found: January 30, 2008
Date found: August 04, 2009
Date found: March 09, 2014
Date found: May 26, 2016
Date found: August 30, 2010
Date found: October 30, 2007
Date found: March 15, 2012
Date found: July 30, 2014
Date found: November 14, 2012
Date found: July 18, 2011
Date found: May 12, 2012
Date found: December 01, 2013
Date found: December 22, 2013
Date found: March 31, 2015
Date found: August 12, 2008
Date found: May 18, 2011
Date found: April 27, 2016
Date found: May 03, 2016
Date found: October 10, 2014
Date found: September 01, 2014
Date found: May 16, 2016
Date found: October 08, 2010
Date found: October 31, 2012
Date found: November 13, 2014
Date found: June 13, 2016
Date found: June 01, 2011
Date found: July 16, 2015
Date found: March 17, 2016
Date found: October 10, 2016
Date found: January 27, 2012
Date found: July 29, 2014
Date found: August 30, 2013
Date found: October 13, 2014
Date found: March 16, 2013
Date found: February 25, 2014
Da

KeyboardInterrupt: 

## Printing the Updated Dataframe

In [28]:
df.head()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,date
0,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,2015-02-04 00:00:00
1,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,2008-01-30 00:00:00
2,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release,2009-08-04 00:00:00
3,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN,2014-03-09 00:00:00
4,12465.json,true,The Chicago Bears have had more starting quart...,education,robin-vos,Wisconsin Assembly speaker,Wisconsin,republican,0.0,3.0,2.0,5.0,1.0,a an online opinion-piece,2016-05-26 00:00:00


## Saving the df as a csv

In [ ]:
df.to_csv('df_with_dates.csv', index=False)
print("CSV file saved successfully!")